In [ ]:
import polars as pl 
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio

# Configuration (après les imports)
pio.renderers.default = "browser"

## 4. Figure 

Objectif global

Calculer et visualiser la proportion de patients retenus dans le traitement au fil du temps,
par bras de traitement (BUPNAL vs CLO)
et par type de patients (IN vs OUT).

Pour chaque jour (VISITNUM) :

- compter combien de patients sont encore présents ( Nt)

- diviser par le nombre initial du bras (N0)

- tracer des points, pas une courbe lissée

différencier :

    - bras (BUPNAL / CLON)
    - type patient (in / out)


| VISITNUM | ARMCD  | patients | proportion |
| -------- | ------ | -------- | ---------- |
| 0        | BUPNAL | in       | 1.00       |
| 1        | BUPNAL | in       | 0.97       |
| …        | CLON   | out      | 0.55       |




#### 4.1 Pipeline 

##### 4.1.1 : Lire DM et EX

In [ ]:
# DM = qui est inclus
dm1 = pl.read_csv("../db/dm1.csv")   
dm2 =pl.read_csv("../db/dm2.csv")
DM = pl.concat([dm1, dm2])

dm = DM.with_columns(
    pl.when(pl.col("STUDYID") == "NIDA-CTN-0001")
      .then(pl.lit("in"))
      .otherwise(pl.lit("out"))
      .alias("patients")
)


##### 4.1.2 : Joindre ARMCD dans EX

In [ ]:
# EX = qui est encore là
ex1 = pl.read_csv("../db/ex1.csv")
ex2 = pl.read_csv("../db/ex2.csv") 
print(ex1.schema)
print(ex2.schema)

| Colonne      | ex1      | ex2       |
| ------------ | -------- | --------- |
| **EXDY**     | `String` | `Int64` ❌ |
| **VISITNUM** | `String` | `Int64` ❌ |


In [ ]:
ex1 = ex1.with_columns(
    pl.col("VISITNUM").cast(pl.Int32, strict=False),
    pl.col("EXDY").cast(pl.Int32, strict=False),
)

ex2 = ex2.with_columns(
    pl.col("VISITNUM").cast(pl.Int32, strict=False),
    pl.col("EXDY").cast(pl.Int32, strict=False),
)
EX = pl.concat([ex1, ex2])


In [ ]:
ex = EX.join(
    dm.select("USUBJID", "ARMCD", "patients"),  # patients = in / out
    on="USUBJID",
)
ex.sample(n=2)

##### 4.1.3 : Garder uniquement BUPNAL et CLON

In [ ]:
ex = ex.filter(
    pl.col("ARMCD").str.contains_any(["BUP", "CLO"])
)
ex.sample(n=2)

##### 4.1.4 : Compter Nt par jour

In [ ]:
df = (
    ex
    .group_by(
        "ARMCD",
        "patients",
        pl.col("VISITNUM").cast(pl.Int32),
        maintain_order=True
    )
    .agg(
        pl.col("USUBJID").unique().len().alias("Nt")
    )
)
df

##### 4.1.5 : Calculer N0 (baseline)

In [ ]:
n0 = (
    dm
    .filter(pl.col("ARMCD").str.contains_any(["BUP", "CLO"]))
    .group_by("ARMCD", "patients")
    .agg(
        pl.col("USUBJID").unique().len().alias("N0")
    )
    .with_columns(VISITNUM=pl.lit(0))
)
n0


##### 4.1.6 : Ajouter le point VISITNUM = 0

In [ ]:
df = pl.concat([
    n0.select("ARMCD", "patients", "VISITNUM", pl.col("N0").alias("Nt")),
    df
])
df

##### 4.1.7 : Calculer la proportion retenue

In [ ]:
df = df.join(
    n0.select("ARMCD", "patients", "N0"),
    on=["ARMCD", "patients"]
).with_columns(
    (pl.col("Nt") / pl.col("N0")).alias("proportion")
)
df

#### 4.2  Tracer Figure 1 

In [ ]:
fig = px.scatter(
    df,
    x="VISITNUM",
    y="proportion",
    color="ARMCD",
    symbol="patients",   # diamant vs carré
    labels={
        "VISITNUM": "Study Days",
        "proportion": "Proportion Retenue",
        "ARMCD": "Arm",
        "patients": ""
    },
    title="Figure 1: Évolution de la rétention des patients au cours du suivi selon le bras de traitement et le profil."
)

fig.update_traces(marker=dict(size=9))
fig.write_html("figure1_retention.html")




Conclusion figure 1 : Cette figure met en évidence une diminution progressive de la rétention des patients au cours du suivi. La rétention est globalement plus élevée chez les patients traités par BUPNAL que par Clonidine, ainsi que chez les patients hospitalisés par rapport aux patients ambulatoires. Ces résultats descriptifs suggèrent un meilleur maintien dans le protocole pour le bras BUPNAL et pour le suivi en milieu hospitalier.

#### 4.4 Conclusion Génerale 

- La Figure 1 montre une meilleure rétention sous BUPNAL par rapport à la Clonidine, chez les patients IN comme OUT.
- Cette tendance est cohérente avec les taux d’abstinence, numériquement plus élevés sous BUPNAL.
- Aucune différence statistiquement significative n’est observée (p-values > 0,05, IC 95 % incluant 0).
- Les écarts observés peuvent donc être attribués au hasard.
- Le déséquilibre des effectifs (ratio 2:1) a probablement réduit la puissance statistique, limitant la détection d’une - - différence réelle.